# Diagnostics: balance sheet violations and statement layoutTwo open questions from `Build_analytics.ipynb`, investigated here rather than inthe build so the deliverable stays a build and this stays an investigation.## 1. Why do 3.48% of filings fail the balance sheet identity?`sum_eiendeler` must equal `sum_egenkapital_gjeld` — total assets equal totalequity plus liabilities. That is an accounting constraint, not an assumptionabout this dataset, yet 15,462 of 444,645 filings violate it above a 0.5 NOKthreshold.Four candidate explanations, and what would distinguish them:| explanation | signature ||---|---|| Our flattening is wrong | MongoDB agrees with itself but disagrees with our table || Units, thousands vs whole NOK | discrepancies cluster on round multiples || One side is internally inconsistent | components of one side fail to sum to its own total || Genuine source error | scattered magnitudes, present identically in MongoDB |The provenance check comes first, because if the fault is ours nothing elsematters. It re-tests the identity directly in MongoDB, server-side, over everyfiling — bypassing Parquet, the schema and the flattening entirely.## 2. What does `oppstillingsplan` tell us, given it never varies?Every filing carries `oppstillingsplan = "store"`. The layout hypothesis formissing revenue is therefore untestable as originally framed: a field with novariance cannot explain variance in anything else.That is a result rather than a dead end, but it leaves the question open, sothe second half tests the same underlying idea using industry code as a proxyfor the kind of income statement a filer keeps, and looks for the positivesignature of a holding or dormant company rather than only the absence ofrevenue.Nothing here writes to the analytics file. Run it after`Build_analytics.ipynb`.

In [ ]:
import json
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pymongo import MongoClient

DATA_DIR = "/home/jovyan/data"
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")
ANALYTICS_FILE = os.path.join(PARQUET_DIR, "analytics_company_financials.parquet")
MONGO_DB = "companiesdb"

spark = SparkSession.builder.appName("group13_diagnostics").getOrCreate()
MONGO_URI = spark.sparkContext.getConf().get("spark.mongodb.read.connection.uri")
db = MongoClient(MONGO_URI)[MONGO_DB]

out = spark.read.parquet(ANALYTICS_FILE)
filed = out.filter("har_regnskap").cache()

# Same rule as the build's verification cell, kept in one place here.
DELTA = F.col("sum_eiendeler") - F.col("sum_egenkapital_gjeld")
TOL = 0.5

bal = filed.filter(F.col("sum_eiendeler").isNotNull()
                   & F.col("sum_egenkapital_gjeld").isNotNull())
viol = bal.filter(F.abs(DELTA) > TOL).cache()

n_bal, n_viol = bal.count(), viol.count()
print("filings with both totals present : %d" % n_bal)
print("violating the identity           : %d  (%.3f%%)" % (n_viol, 100.0 * n_viol / n_bal))

findings = {"tolerance_nok": TOL, "checked": n_bal, "violations": n_viol,
            "violation_pct": 100.0 * n_viol / n_bal}

## Provenance: is this ours or the source's?The identity is re-tested inside MongoDB, server-side, over every filing. Thispath shares nothing with the analytics table — not Parquet, not `schemas.py`,not the flattening — so agreement means the discrepancy arrived with the dataand disagreement means we introduced it.Counted over all filings, not a sample.

In [ ]:
pipeline = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$project": {
        "a": "$data.eiendeler.sumEiendeler",
        "b": "$data.egenkapitalGjeld.sumEgenkapitalGjeld",
    }},
    {"$match": {"a": {"$ne": None}, "b": {"$ne": None}}},
    {"$group": {
        "_id": None,
        "n": {"$sum": 1},
        "viol": {"$sum": {"$cond": [
            {"$gt": [{"$abs": {"$subtract": ["$a", "$b"]}}, TOL]}, 1, 0]}},
    }},
]
mongo_res = db.financial_data.aggregate(pipeline, allowDiskUse=True).next()

print("MongoDB, both totals present : %d" % mongo_res["n"])
print("MongoDB, violating           : %d  (%.3f%%)"
      % (mongo_res["viol"], 100.0 * mongo_res["viol"] / mongo_res["n"]))
print()
agrees = (mongo_res["n"] == n_bal) and (mongo_res["viol"] == n_viol)
print("Agrees with the analytics table: %s" % ("YES" if agrees else "NO"))
if agrees:
    print("  The discrepancy is present in the source documents. Our Parquet")
    print("  export and flattening did not introduce it.")
else:
    print("  The two disagree. The fault is somewhere in export or flattening,")
    print("  and everything below is measuring our own bug rather than the data.")

findings["mongodb"] = {"checked": mongo_res["n"], "violations": mongo_res["viol"],
                       "agrees_with_analytics": agrees}

## MagnitudeAbsolute discrepancy in NOK, and relative to the size of the balance sheet.The relative figure is what decides whether this matters. A 100 NOK gap on abillion-NOK balance sheet is noise; the same gap on a 500 NOK balance sheet isthe whole statement.

In [ ]:
mag = viol.select(
    F.abs(DELTA).alias("abs_delta"),
    DELTA.alias("signed_delta"),
    F.col("sum_eiendeler").alias("assets"),
    (F.abs(DELTA) / F.greatest(F.abs(F.col("sum_eiendeler")), F.lit(1.0)) * 100).alias("rel_pct"),
).cache()

probs = [0.01, 0.25, 0.5, 0.75, 0.9, 0.99]
labels = ["p1", "p25", "median", "p75", "p90", "p99"]

abs_q = mag.approxQuantile("abs_delta", probs, 0.0)
rel_q = mag.approxQuantile("rel_pct", probs, 0.0)

print("%-8s %18s %14s" % ("", "abs delta (NOK)", "rel to assets"))
print("-" * 44)
for lab, a, r in zip(labels, abs_q, rel_q):
    print("%-8s %18.2f %13.4f%%" % (lab, a, r))

ext = mag.agg(F.min("abs_delta"), F.max("abs_delta"),
              F.min("rel_pct"), F.max("rel_pct")).collect()[0]
print("%-8s %18.2f %13.4f%%" % ("min", ext[0], ext[2]))
print("%-8s %18.2f %13.4f%%" % ("max", ext[1], ext[3]))

# Direction: a consistent sign would point at a systematic omission on one side.
n_pos = mag.filter("signed_delta > 0").count()
print()
print("assets exceed equity+liabilities : %d  (%.1f%%)" % (n_pos, 100.0 * n_pos / n_viol))
print("equity+liabilities exceed assets : %d  (%.1f%%)"
      % (n_viol - n_pos, 100.0 * (n_viol - n_pos) / n_viol))

findings["magnitude"] = {
    "abs_delta_quantiles": dict(zip(labels, abs_q)),
    "rel_pct_quantiles": dict(zip(labels, rel_q)),
    "abs_delta_min": float(ext[0]), "abs_delta_max": float(ext[1]),
    "assets_exceed_liabilities": n_pos,
    "liabilities_exceed_assets": n_viol - n_pos,
}

## Units testIf the register mixed whole NOK with thousands, the discrepancies would land onround multiples far more often than chance allows. Scattered residues point atsomething else.The reference point matters: about 0.1% of arbitrary values are multiples of1000 by chance, so anything near that rate is not evidence of a units problem.

In [ ]:
rounded = mag.select(
    (F.col("abs_delta") == F.round("abs_delta")).alias("whole"),
    (F.col("abs_delta") % 100 == 0).alias("mult_100"),
    (F.col("abs_delta") % 1000 == 0).alias("mult_1000"),
    (F.col("abs_delta") % 1000000 == 0).alias("mult_1e6"),
).agg(*[F.sum(F.col(c).cast("long")).alias(c)
        for c in ["whole", "mult_100", "mult_1000", "mult_1e6"]]).collect()[0]

print("%-14s %10s %9s   %s" % ("multiple of", "count", "share", "expected by chance"))
print("-" * 62)
for col, chance in [("whole", "n/a"), ("mult_100", "1.0%"),
                    ("mult_1000", "0.1%"), ("mult_1e6", "0.0001%")]:
    print("%-14s %10d %8.2f%%   %s"
          % (col, rounded[col], 100.0 * rounded[col] / n_viol, chance))

findings["rounding"] = {c: int(rounded[c]) for c in
                        ["whole", "mult_100", "mult_1000", "mult_1e6"]}

## Which side is internally inconsistent?Each side of the balance sheet has its own components, so each can be checkedagainst its own total independently of the other:- assets: `omloepsmidler + anleggsmidler` should equal `sum_eiendeler`- equity and liabilities: `sum_egenkapital + sum_gjeld` should equal  `sum_egenkapital_gjeld`If one side sums correctly and the other does not, the fault is localised. Ifboth sum correctly, then each side is internally coherent and they simplydisagree with each other, which points at the source rather than at a droppedcomponent.Only rows where every component is present are counted, so a null is neversilently read as a zero.

In [ ]:
assets_ok = (F.abs(F.col("omloepsmidler") + F.col("anleggsmidler")
                   - F.col("sum_eiendeler")) <= TOL)
liab_ok = (F.abs(F.col("sum_egenkapital") + F.col("sum_gjeld")
                 - F.col("sum_egenkapital_gjeld")) <= TOL)

a_full = viol.filter(F.col("omloepsmidler").isNotNull()
                     & F.col("anleggsmidler").isNotNull())
l_full = viol.filter(F.col("sum_egenkapital").isNotNull()
                     & F.col("sum_gjeld").isNotNull())

n_a, n_a_ok = a_full.count(), a_full.filter(assets_ok).count()
n_l, n_l_ok = l_full.count(), l_full.filter(liab_ok).count()

print("Among the %d violating filings:" % n_viol)
print("  assets side components present   : %d" % n_a)
print("    and summing to their own total : %d  (%.1f%%)"
      % (n_a_ok, 100.0 * n_a_ok / n_a if n_a else 0))
print("  equity/liab components present   : %d" % n_l)
print("    and summing to their own total : %d  (%.1f%%)"
      % (n_l_ok, 100.0 * n_l_ok / n_l if n_l else 0))

# Same check on conforming filings, as a control. Without it there is no way to
# tell whether a low rate is characteristic of the violators or of the corpus.
ok_rows = bal.filter(F.abs(DELTA) <= TOL)
c_a = ok_rows.filter(F.col("omloepsmidler").isNotNull() & F.col("anleggsmidler").isNotNull())
c_l = ok_rows.filter(F.col("sum_egenkapital").isNotNull() & F.col("sum_gjeld").isNotNull())
c_a_n, c_a_ok = c_a.count(), c_a.filter(assets_ok).count()
c_l_n, c_l_ok = c_l.count(), c_l.filter(liab_ok).count()

print()
print("Control, the %d conforming filings:" % ok_rows.count())
print("  assets side sums correctly       : %d of %d  (%.1f%%)"
      % (c_a_ok, c_a_n, 100.0 * c_a_ok / c_a_n if c_a_n else 0))
print("  equity/liab side sums correctly  : %d of %d  (%.1f%%)"
      % (c_l_ok, c_l_n, 100.0 * c_l_ok / c_l_n if c_l_n else 0))

findings["side_consistency"] = {
    "violating": {"assets_checked": n_a, "assets_ok": n_a_ok,
                  "liab_checked": n_l, "liab_ok": n_l_ok},
    "conforming": {"assets_checked": c_a_n, "assets_ok": c_a_ok,
                   "liab_checked": c_l_n, "liab_ok": c_l_ok},
}

## Does it cluster?Rates are reported against each group's own base, not as a share of allviolations, so a large legal form does not look like a culprit merely for beinglarge.

In [ ]:
def rate_by(column, limit=12):
    """Violation rate within each value of `column`, largest groups first."""
    marked = bal.withColumn("_viol", (F.abs(DELTA) > TOL).cast("int"))
    rows = (marked.groupBy(column)
            .agg(F.count(F.lit(1)).alias("n"), F.sum("_viol").alias("viol"))
            .orderBy(F.desc("n")).limit(limit).collect())
    print("%-22s %10s %10s %9s" % (column, "filings", "violations", "rate"))
    print("-" * 54)
    result = {}
    for r in rows:
        key = str(r[column])
        pct = 100.0 * r["viol"] / r["n"]
        result[key] = {"n": r["n"], "violations": r["viol"], "pct": pct}
        print("%-22s %10d %10d %8.2f%%" % (key, r["n"], r["viol"], pct))
    print()
    return result


findings["clustering"] = {}
for col in ["organisasjonsform_kode", "regnskapsaar", "valuta",
            "smaa_foretak", "avviklingsregnskap", "is_full_year",
            "morselskap", "ikke_revidert"]:
    findings["clustering"][col] = rate_by(col)

## SamplesTen violating filings, spanning the magnitude range, so a few can be checked byhand against the Regnskapsregisteret API. Nothing here proves anything on itsown; it is material for a manual spot check.

In [ ]:
sample = (viol.select("organisasjonsnummer", "navn", "organisasjonsform_kode",
                      "regnskapsaar", "sum_eiendeler", "sum_egenkapital_gjeld",
                      "sum_egenkapital", "sum_gjeld",
                      DELTA.alias("delta"))
          .orderBy(F.desc(F.abs(DELTA))).limit(5))
print("Largest discrepancies:")
sample.show(5, truncate=30)

print("Smallest discrepancies above tolerance:")
(viol.select("organisasjonsnummer", "navn", "sum_eiendeler",
             "sum_egenkapital_gjeld", DELTA.alias("delta"))
     .orderBy(F.abs(DELTA)).limit(5).show(5, truncate=30))

findings["samples"] = [r.asDict() for r in sample.collect()]
print("API check: https://data.brreg.no/regnskapsregisteret/regnskap/<organisasjonsnummer>")

# Part 2: statement layout## Is `oppstillingsplan` really constant?Checked at all three layers. If it varies in MongoDB but not in our table, weflattened it wrongly; if it is constant everywhere, the API genuinely returnsone layout for every filer.

In [ ]:
print("Analytics table:")
for r in filed.groupBy("oppstillingsplan").count().orderBy(F.desc("count")).collect():
    print("   %-16s %d" % (r["oppstillingsplan"], r["count"]))

print("\nRaw Parquet mirror:")
raw = spark.read.parquet(os.path.join(PARQUET_DIR, "financial_data")).filter(
    "fetch_status = 'success'")
for r in (raw.groupBy(F.col("data")[0]["oppstillingsplan"].alias("plan"))
          .count().orderBy(F.desc("count")).collect()):
    print("   %-16s %d" % (r["plan"], r["count"]))

print("\nMongoDB:")
for r in db.financial_data.aggregate([
        {"$match": {"fetch_status": "success"}},
        {"$unwind": "$data"},
        {"$group": {"_id": "$data.oppstillingsplan", "n": {"$sum": 1}}},
        {"$sort": {"n": -1}}], allowDiskUse=True):
    print("   %-16s %d" % (r["_id"], r["n"]))

# regnskapsregler is the other field that could distinguish accounting regimes.
print("\nregnskapsregler, the other candidate discriminator:")
for r in (filed.groupBy("regnskapsregler").count()
          .orderBy(F.desc("count")).collect()):
    print("   %-16s %d" % (r["regnskapsregler"], r["count"]))

findings["oppstillingsplan"] = {
    "analytics": {r["oppstillingsplan"]: r["count"] for r in
                  filed.groupBy("oppstillingsplan").count().collect()},
    "regnskapsregler": {r["regnskapsregler"]: r["count"] for r in
                        filed.groupBy("regnskapsregler").count().collect()},
}

## Industry as a proxy for the income statementThe layout hypothesis was that banks and insurers keep a different incomestatement with no `driftsinntekter` line. `oppstillingsplan` cannot test it, butindustry code can: NACE divisions 64 to 66 are financial and insuranceactivities, and 64.20 specifically is holding companies.This also separates the two hypotheses, which the original framing conflated.A *bank* has operating revenue and reports it on a different line; a *holdingcompany* has no operating revenue at all. Both would show as `revenue_missing`,but only the second is a dormant-entity story.

In [ ]:
div = F.substring(F.col("naeringskode1_kode"), 1, 2)
missing = (F.col("operating_margin_status") == "revenue_missing").cast("int")

by_div = (filed.withColumn("division", div)
          .groupBy("division")
          .agg(F.count(F.lit(1)).alias("n"), F.sum(missing).alias("miss"))
          .filter("n >= 500")
          .withColumn("pct", 100.0 * F.col("miss") / F.col("n")))

base = 100.0 * filed.agg(F.sum(missing)).collect()[0][0] / filed.count()
print("Corpus-wide revenue_missing rate: %.2f%%" % base)

print("\nHighest revenue_missing rate by NACE division (min 500 filings):")
print("%-10s %10s %10s %9s" % ("division", "filings", "missing", "rate"))
print("-" * 42)
top_div = by_div.orderBy(F.desc("pct")).limit(15).collect()
for r in top_div:
    print("%-10s %10d %10d %8.2f%%" % (r["division"], r["n"], r["miss"], r["pct"]))

# Named divisions, so the report can point at them rather than at bare codes.
NAMED = {"64": "Financial service activities (incl. holding companies)",
         "65": "Insurance and pension funding",
         "66": "Activities auxiliary to financial services",
         "68": "Real estate activities",
         "70": "Head office and management consultancy"}
print("\nDivisions of specific interest:")
named_rows = by_div.filter(F.col("division").isin(list(NAMED))).collect()
for r in sorted(named_rows, key=lambda x: -x["pct"]):
    print("  %-4s %-52s %8.2f%%  (n=%d)"
          % (r["division"], NAMED[r["division"]], r["pct"], r["n"]))

findings["revenue_missing_by_division"] = {
    "corpus_rate_pct": base,
    "top15": {r["division"]: {"n": r["n"], "miss": r["miss"], "pct": r["pct"]}
              for r in top_div},
    "named": {r["division"]: {"n": r["n"], "miss": r["miss"], "pct": r["pct"]}
              for r in named_rows},
}

## The positive signature of a dormant or holding companyAbsence of revenue is weak evidence on its own. Three things should be true ifthese are holding and dormant entities rather than filers using another layout:1. they employ nobody2. their result comes from financial rather than operating income3. they still hold a balance sheetThe employee comparison is the sharpest of the three, because a bank has staffand a shell company does not.

In [ ]:
miss_rows = filed.filter("operating_margin_status = 'revenue_missing'")
comp_rows = filed.filter("operating_margin_status = 'computed'")

def employee_profile(df, label):
    n = df.count()
    zero_or_null = df.filter(F.col("antall_ansatte").isNull()
                             | (F.col("antall_ansatte") == 0)).count()
    med = df.filter(F.col("antall_ansatte").isNotNull()) \
            .approxQuantile("antall_ansatte", [0.5], 0.0)
    print("  %-18s n=%7d   no employees %6.2f%%   median (where recorded) %s"
          % (label, n, 100.0 * zero_or_null / n, med[0] if med else "n/a"))
    return {"n": n, "no_employees_pct": 100.0 * zero_or_null / n,
            "median_employees": med[0] if med else None}

print("Employees:")
emp = {"revenue_missing": employee_profile(miss_rows, "revenue_missing"),
       "computed": employee_profile(comp_rows, "computed")}

# If income is financial rather than operating, netto_finans should be present
# and material far more often among the revenue_missing group.
print("\nFinancial income:")
fin_prof = {}
for df, label in [(miss_rows, "revenue_missing"), (comp_rows, "computed")]:
    n = df.count()
    has_fin = df.filter(F.col("netto_finans").isNotNull()
                        & (F.abs(F.col("netto_finans")) > 0)).count()
    has_bal = df.filter(F.col("sum_eiendeler").isNotNull()
                        & (F.col("sum_eiendeler") > 0)).count()
    fin_prof[label] = {"n": n,
                       "nonzero_netto_finans_pct": 100.0 * has_fin / n,
                       "nonzero_balance_sheet_pct": 100.0 * has_bal / n}
    print("  %-18s non-zero netto_finans %6.2f%%   non-zero balance sheet %6.2f%%"
          % (label, 100.0 * has_fin / n, 100.0 * has_bal / n))

findings["dormant_signature"] = {"employees": emp, "financial": fin_prof}

## Persist

In [ ]:
path = os.path.join(DATA_DIR, "diagnostics_balance_and_layout.json")
with open(path, "w") as fh:
    json.dump(findings, fh, indent=2, default=str)

print("Wrote", path)
for k in findings:
    print("  ", k)

## How to read the output**Balance violations.** If the MongoDB check agrees with the analytics table,the discrepancy is in the source and the rest of the section characterises it.If it disagrees, stop: the fault is in our export or flattening and every figurebelow is measuring our own bug.Then, in order: a high multiple-of-1000 rate well above the 0.1% chancebaseline indicates a units problem. One side failing to sum to its own totalwhile the other succeeds localises the fault to that side. Both sides summingcorrectly while disagreeing with each other means the source is internallycoherent and simply wrong. Concentration in `avviklingsregnskap` or a single`regnskapsaar` would point at a specific filing regime rather than a generaldefect.**Layout.** If `oppstillingsplan` is constant at all three layers, the APIreturns one layout for every filer and the field carries no information. Thedivision breakdown then tests the real question: a high `revenue_missing` ratein divisions 64 to 66 supports the different-income-statement explanation, whilea rate that tracks holding-company divisions and zero-employee entities insteadsupports the dormant-company one. The two are not mutually exclusive, and theemployee comparison is what separates them.